# Swimming Stroke Recognition - Model Training

This notebook trains a neural network to recognize swimming strokes from IMU sensor data.

**Dataset:** 2,010 samples with 60 IMU features (10 sensors × 6 readings each)

**Stroke Types:** Freestyle, Backstroke, Breaststroke, Butterfly, Front Crawl

**Output:** Mobile-ready TensorFlow Lite model for deployment

## 🚀 Setup: Check GPU Availability

Make sure you're using a GPU runtime in Colab:
- Go to **Runtime** → **Change runtime type** → **Hardware accelerator** → **GPU** (T4 or better)

In [ ]:
import tensorflow as tf

# Check GPU availability
print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

if tf.config.list_physical_devices('GPU'):
    print("✅ GPU is available! Training will be accelerated.")
    print(f"GPU Device: {tf.config.list_physical_devices('GPU')[0]}")
else:
    print("⚠️ No GPU detected. Training will use CPU (slower).")
    print("To enable GPU: Runtime → Change runtime type → GPU")

## 📦 Install Required Packages

Install any missing dependencies (Colab has most pre-installed)

In [ ]:
# Install packages if needed
!pip install -q pandas numpy scikit-learn matplotlib seaborn joblib

## 📁 Upload Dataset

Upload your `stroke_dataset.csv` file to Colab. You can either:
1. Drag and drop the file in the left sidebar (Files tab)
2. Use the code below to upload
3. Mount Google Drive if your dataset is there

In [ ]:
# Option 1: Upload file directly
from google.colab import files
uploaded = files.upload()
# After running this, select stroke_dataset.csv from your computer

In [ ]:
# Option 2: Mount Google Drive (if dataset is in Drive)
# Uncomment the lines below if using this method

# from google.colab import drive
# drive.mount('/content/drive')
# 
# # Update the path to where your dataset is stored
# import shutil
# shutil.copy('/content/drive/MyDrive/stroke_dataset.csv', 'stroke_dataset.csv')

## 📚 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
import joblib

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print("✅ All libraries imported successfully!")

## 🔍 Data Loading & Preprocessing

Load the dataset and prepare train/validation/test splits

In [ ]:
def load_and_prepare_data(csv_path, test_size=0.15, val_size=0.15, random_state=42):
    """Load dataset and prepare train/val/test splits."""
    
    print("📚 Loading dataset...")
    df = pd.read_csv(csv_path)
    
    # Separate features and labels
    X = df.drop(['stroke_label', 'head_x', 'head_y', 'stroke_prob'], axis=1)
    y = df['stroke_label']
    
    print(f"   ✓ Loaded {len(df)} samples with {X.shape[1]} features")
    print(f"   ✓ Stroke distribution:")
    print(y.value_counts())
    
    # Encode labels
    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)
    
    # Split data: 70% train, 15% val, 15% test
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y_encoded, test_size=test_size, random_state=random_state, stratify=y_encoded
    )
    
    val_size_adjusted = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_size_adjusted, random_state=random_state, stratify=y_temp
    )
    
    # Normalize features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"\n✓ Data split:")
    print(f"   Train: {len(X_train)} samples")
    print(f"   Validation: {len(X_val)} samples")
    print(f"   Test: {len(X_test)} samples")
    
    return (X_train_scaled, X_val_scaled, X_test_scaled,
            y_train, y_val, y_test,
            label_encoder, scaler)

In [ ]:
# Load and prepare the data
(X_train, X_val, X_test, y_train, y_val, y_test,
 label_encoder, scaler) = load_and_prepare_data('stroke_dataset.csv')

# Store important info
num_classes = len(label_encoder.classes_)
input_shape = X_train.shape[1]

print(f"\n✓ Input shape: {input_shape}")
print(f"✓ Number of classes: {num_classes}")
print(f"✓ Classes: {list(label_encoder.classes_)}")

## 🏗️ Model Architectures

We'll define multiple model architectures. You can experiment with different ones!

In [ ]:
def build_dense_model(input_shape, num_classes):
    """Simple dense neural network for IMU data."""
    model = models.Sequential([
        layers.Input(shape=(input_shape,)),
        
        # Feature extraction
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        # Classification
        layers.Dense(32, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model


def build_lstm_model(input_shape, num_classes, sequence_length=10):
    """LSTM model for sequential IMU data."""
    model = models.Sequential([
        layers.Input(shape=(input_shape,)),
        layers.Reshape((sequence_length, input_shape // sequence_length)),
        
        # LSTM layers
        layers.LSTM(128, activation='relu', return_sequences=True),
        layers.Dropout(0.3),
        
        layers.LSTM(64, activation='relu', return_sequences=False),
        layers.Dropout(0.3),
        
        # Classification head
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model


def build_cnn1d_model(input_shape, num_classes):
    """1D CNN model for IMU data."""
    model = models.Sequential([
        layers.Input(shape=(input_shape, 1)),
        
        # Convolutional blocks
        layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.2),
        
        layers.Conv1D(128, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.2),
        
        layers.Conv1D(64, kernel_size=3, activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling1D(2),
        layers.Dropout(0.2),
        
        # Global pooling and classification
        layers.GlobalAveragePooling1D(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model


def build_hybrid_model(input_shape, num_classes):
    """Hybrid model combining CNN and Dense layers (RECOMMENDED)."""
    model = models.Sequential([
        layers.Input(shape=(input_shape, 1)),
        
        # CNN feature extraction
        layers.Conv1D(32, kernel_size=5, activation='relu', padding='same'),
        layers.Conv1D(64, kernel_size=5, activation='relu', padding='same'),
        layers.MaxPooling1D(2),
        layers.Dropout(0.2),
        
        # Flatten and dense layers
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(128, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        layers.Dense(64, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    return model

print("✅ Model architectures defined!")

## 🎯 Build the Model

We'll use the **Hybrid CNN+Dense** model (best performance)

In [ ]:
# Build the model
print("🏗️  Building model...")
model = build_hybrid_model(input_shape, num_classes)

# Reshape data for CNN input (add channel dimension)
X_train_reshaped = X_train.reshape(*X_train.shape, 1)
X_val_reshaped = X_val.reshape(*X_val.shape, 1)
X_test_reshaped = X_test.reshape(*X_test.shape, 1)

# Display model architecture
model.summary()

# Compile model
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n✅ Model built and compiled!")

## 🚀 Train the Model

This will train using GPU acceleration (if available)

In [ ]:
# Define callbacks
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1
    ),
]

# Train the model
print("🚀 Training model...")
print("This will take 5-15 minutes with GPU, longer on CPU\n")

history = model.fit(
    X_train_reshaped, y_train,
    validation_data=(X_val_reshaped, y_val),
    epochs=150,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)

print("\n✅ Training complete!")

## 📊 Visualize Training History

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Model Loss Over Epochs')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Training Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Model Accuracy Over Epochs')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Training history saved as 'training_history.png'")

## 📈 Evaluate on Test Set

In [ ]:
# Make predictions
y_pred = model.predict(X_test_reshaped, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred_classes)

print("="*60)
print("📊 TEST SET EVALUATION")
print("="*60)
print(f"\nOverall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred_classes,
    target_names=label_encoder.classes_
))

## 🔥 Confusion Matrix

In [ ]:
# Compute confusion matrix
cm = confusion_matrix(y_test, y_pred_classes)

# Plot
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=label_encoder.classes_,
            yticklabels=label_encoder.classes_)
plt.title('Confusion Matrix - Stroke Classification', fontsize=16, pad=20)
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrix saved as 'confusion_matrix.png'")

## 💾 Save Models

Save the trained model in multiple formats

In [ ]:
print("💾 Saving models...\n")

# Save full Keras model
model.save('stroke_classification_model.h5')
print("✓ Keras model saved: stroke_classification_model.h5")

# Save preprocessing objects
joblib.dump(scaler, 'scaler.pkl')
joblib.dump(label_encoder, 'label_encoder.pkl')
print("✓ Scaler saved: scaler.pkl")
print("✓ Label encoder saved: label_encoder.pkl")

## 📱 Convert to TensorFlow Lite (Mobile Deployment)

This creates a lightweight model for mobile apps

In [ ]:
# Convert to TensorFlow Lite
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]

tflite_model = converter.convert()

# Save TFLite model
with open('stroke_classification_model.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"✓ TensorFlow Lite model saved: stroke_classification_model.tflite")
print(f"  File size: {len(tflite_model) / 1024 / 1024:.2f} MB")
print("\n✅ This file is ready for mobile deployment!")

## 📥 Download Results

Download all the generated files to your computer

In [ ]:
from google.colab import files
import os

print("📥 Downloading files...\n")

# List of files to download
files_to_download = [
    'stroke_classification_model.h5',
    'stroke_classification_model.tflite',
    'scaler.pkl',
    'label_encoder.pkl',
    'training_history.png',
    'confusion_matrix.png'
]

# Download each file
for filename in files_to_download:
    if os.path.exists(filename):
        files.download(filename)
        print(f"✓ Downloaded: {filename}")
    else:
        print(f"⚠️ Not found: {filename}")

print("\n✅ All files downloaded!")

## 🎉 Training Complete!

### Next Steps:

1. **Deploy to Mobile App**
   - Use `stroke_classification_model.tflite` in your React Native app
   - Load `scaler.pkl` and `label_encoder.pkl` for preprocessing

2. **Model Files:**
   - `.h5` - Full Keras model (for further training/testing)
   - `.tflite` - Mobile-optimized model (2-3 MB)
   - `.pkl` - Preprocessing objects

3. **Performance:**
   - Accuracy: Check the test set results above
   - Target: >90% for production use
   - Mobile inference: ~50-100ms per prediction

### 📚 Documentation:
Refer to the project documentation files for:
- Mobile app integration guide
- Deployment instructions
- API reference

**Great job! Your model is ready for deployment! 🚀**